In [1]:
# Core analysis tools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import sys
import os

# Add repo root to path
sys.path.insert(
    0,
    os.path.abspath("..")
)

# Custom analysis modules
from src.io import load_samples
from src import preprocessing
from src import dim_reduction as dr
from src import expression
from src import markers
from src import annotation
from src import gep
from src import plotting

## Get Processed Counts from CellRanger output and Analyze Expression

Select a parent directory, then loop across the subdirectories to extract the single-cell matrices and compare using Scanpy. Subdirectory names should correspond to condition. Then, concatenate across samples.

Here, manually input the path and sample names for each run.

In [ ]:
parent_dir = "/Users/norawolcott/Documents/Datta Lab/data/scRNA-seq/260706"
samples = ["D1", "D2", "P1", "P2"]

adata_all = load_samples(parent_dir, samples)

adata_all.obs["condition"] = (
    adata_all.obs["sample"]
    .map({
        "D1": "Diestrus",
        "D2": "Diestrus",
        "P1": "Proestrus",
        "P2": "Proestrus"
    })
)

Loading /Users/norawolcott/Documents/Datta Lab/data/scRNA-seq/260706/D1/MOE_female_D1_counts.h5ad
Loading /Users/norawolcott/Documents/Datta Lab/data/scRNA-seq/260706/D2/MOE_female_D2_counts.h5ad
Loading /Users/norawolcott/Documents/Datta Lab/data/scRNA-seq/260706/P1/MOE_female_P1_counts.h5ad
Loading /Users/norawolcott/Documents/Datta Lab/data/scRNA-seq/260706/P2/MOE_female_P2_counts.h5ad


Find the most differentially expressed genes across cells

In [150]:
# Store raw counts
adata_all = preprocessing.store_raw_counts(adata_all)

# Normalize
adata_all = preprocessing.normalize_data(adata_all)

# Select HVGs
adata_all = preprocessing.identify_hvgs(
    adata_all,
    n_top_genes=3000
)

top10_hvg = (
    adata_all.var[adata_all.var["highly_variable"]]
    .sort_values("dispersions_norm", ascending=False)
    .head(10)
)

print(top10_hvg.index.tolist())

['Bpifb3', 'Hba-a1', 'Reg3g', 'Wfdc18', 'Hbb-bt', 'S100a8', 'S100a9', 'Coch', 'Apod', 'Vmo1']


## Dimensionality Reduction
Perform PCA, KNN, UMAP, and Leiden analysis.

In [151]:
# PCA
adata_all = dr.run_pca(adata_all)

variance = dr.get_pca_variance(adata_all)

print("Variance explained:")
for i, v in enumerate(variance[:10]):
    print(f"PC{i+1}: {v*100:.2f}%")

print(
    f"\nTotal variance explained by first 10 PCs: "
    f"{variance[:10].sum()*100:.2f}%"
)

# Dimensionality reduction
adata_all = dr.run_umap(
    adata_all,
    n_pcs=35
)

adata_all = dr.leiden_clustering(
    adata_all,
    resolution=0.5
)

# Plot embeddings
plotting.plot_umap_metadata(adata_all)
plotting.plot_umap_clusters(adata_all)


/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/dim_reduction.py:12: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  n_comps : int, optional


Variance explained:
PC1: 16.50%
PC2: 9.96%
PC3: 3.38%
PC4: 3.23%
PC5: 2.89%
PC6: 2.34%
PC7: 2.14%
PC8: 1.85%
PC9: 1.84%
PC10: 1.55%

Total variance explained by first 10 PCs: 45.69%


/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/dim_reduction.py:49: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
  Annotated single-cell dataset.


Analyze expression of HVGs and top marker genes.

In [152]:
# Find marker genes
adata_all = expression.rank_cluster_markers(
    adata_all,
    groupby="leiden"
)


# Get top markers
top_genes = expression.get_top_marker_genes(
    adata_all
)

print("Top DE gene for each Leiden group:")
for group, genes in top_genes.items():
    print(f"  {group}: {genes[0]}")

Top DE gene for each Leiden group:
  0: Pcp4l1
  1: Nqo1
  2: H2bc21
  3: Gap43
  4: Uchl1
  5: Scn9a
  6: Calb2
  7: Cd36
  8: Crabp1
  9: Cbr2
  10: Hnrnpa1
  11: Rn18s-rs5
  12: Cyp2a5
  13: Moxd1
  14: Laptm5
  15: Alox5ap
  16: Sox11
  17: Sparc


## Identify mature OSNs

Score data according to marker genes (specified in markers.py).

In [153]:
marker_sets = {
    "mature_OSN_score": markers.mature_osn_markers,
    "immature_OSN_score": markers.immature_osn_markers,
    "non_neuronal_score": markers.non_neuronal_markers,
    "osn_score": markers.osn_markers,
    "epithelial_score": markers.epithelial_markers
}

adata_all = annotation.score_gene_sets(
    adata_all,
    marker_sets
)

Plot UMAPs of marker genes

In [154]:
# Plot OSN scoring metrics
plotting.plot_score_umap(
    adata_all
)

plotting.plot_marker_umap(
    adata_all,
    genes=markers.osn_markers,
    cmap="viridis_r",
    filename="osn_marker_umaps.png"
)

plotting.plot_marker_umap(
    adata_all,
    genes=markers.epithelial_markers,
    cmap="viridis_r",
    filename="epithelial_marker_umaps.png"
)

Identify mature OSN clusters

In [155]:
likely_mature_osns, cluster_scores = annotation.identify_mature_osn_clusters(
    adata_all
)

Cluster score summary:
        mature_OSN_score  immature_OSN_score  non_neuronal_score  \
leiden                                                             
0               1.613843           -0.289776           -0.095602   
1               1.556506           -0.175442           -0.089391   
2               1.525221           -0.269439           -0.093501   
3               0.129937            0.962772           -0.121407   
4               1.327626            0.061864           -0.112245   
5               1.683582           -0.201314           -0.083188   
6               1.631350            0.193393           -0.098945   
7               1.430513           -0.277894           -0.060383   
8              -0.732099            1.093004           -0.091141   
9              -0.455530           -0.173409            1.721669   
10             -0.715738            0.399281            0.069763   
11              0.439123           -0.027145            0.402133   
12              0.126465 

/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/annotation.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby(cluster_key)[


Check OR expression

In [156]:
adata_all = plotting.plot_or_gene_counts(
    adata_all
)

Identify OSN-OSN doublets

In [157]:
candidate_osn_osn_doublets, cluster_doublet_screen = annotation.screen_osn_doublets(
    adata_all
)

print("Candidate OSN-OSN doublet clusters:")
print(candidate_osn_osn_doublets)

display(
    cluster_doublet_screen.sort_values(
        "total_counts",
        ascending=False
    )
)

Candidate OSN-OSN doublet clusters:
['16']


/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/annotation.py:70: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby(cluster_key)[
/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/annotation.py:82: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby(cluster_key)["n_ORs"]


,total_counts,n_genes_by_counts,n_ORs,pct_cells_multiple_ORs
leiden,,,,
16,22587.183594,5894.361386,6.004950,0.777228
10,22528.035156,5218.012346,0.739369,0.146776
12,19844.923828,4799.777778,2.173611,0.488426
4,16989.882812,4953.490973,1.828710,0.521356
3,15859.238281,4990.577836,1.853122,0.471416
6,12402.009766,4112.167598,1.649907,0.469739
0,11867.175781,3936.382987,1.867161,0.582987
9,11516.637695,3612.581773,0.802747,0.171036
8,11470.171875,3847.702247,5.962279,0.582665


Identify OSN + non-neuronal doublets

In [158]:
candidate_osn_nonneuronal_doublets, cluster_mixed_screen = (
    annotation.screen_osn_nonneuronal_doublets(
        adata_all
    )
)

print("Candidate OSN + non-neuronal doublet clusters:")
print(candidate_osn_nonneuronal_doublets)

display(
    cluster_mixed_screen.sort_values(
        "mature_OSN_score",
        ascending=False
    )
)

Candidate OSN + non-neuronal doublet clusters:
['5', '7', '11']


/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/annotation.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby(cluster_key)[


,mature_OSN_score,non_neuronal_score,immature_OSN_score,total_counts,pct_counts_mito
leiden,,,,,
5,1.683582,-0.083188,-0.201314,5863.077637,5.144678
6,1.631350,-0.098945,0.193393,12402.009766,1.848110
0,1.613843,-0.095602,-0.289776,11867.175781,2.079786
1,1.556506,-0.089391,-0.175442,10521.634766,1.797655
2,1.525221,-0.093501,-0.269439,10691.076172,1.844417
7,1.430513,-0.060383,-0.277894,11043.298828,1.730054
4,1.327626,-0.112245,0.061864,16989.882812,1.355401
16,1.033962,-0.113624,0.738446,22587.183594,1.930050
11,0.439123,0.402133,-0.027145,3836.128662,18.260817


Remove OSN doublet clusters from mature OSN candidates

In [159]:
# Combine all candidate doublet clusters
all_candidate_doublets = (
    candidate_osn_osn_doublets +
    candidate_osn_nonneuronal_doublets
)

# Remove doublet clusters from mature OSN candidates
likely_mature_osns_filtered = annotation.filter_doublet_clusters(
    likely_mature_osns,
    all_candidate_doublets
)

print("Original likely mature OSN clusters:")
print(likely_mature_osns)

print("\nRemoving candidate doublet clusters:")
print(all_candidate_doublets)

print("\nFinal mature OSN clusters:")
print(likely_mature_osns_filtered)

Original likely mature OSN clusters:
['0', '1', '2']

Removing candidate doublet clusters:
['16', '5', '7', '11']

Final mature OSN clusters:
['0', '1', '2']


Recalculate HVGs just using mature OSN clusters

In [160]:
adata_mature_osn = annotation.subset_clusters(
    adata_all,
    likely_mature_osns_filtered
)

print(f"Starting mature OSN cells: {adata_mature_osn.n_obs}")

Starting mature OSN cells: 18735


Identify mitochondrial genes

In [161]:
adata_mature_osn = preprocessing.filter_cells_by_qc(
    adata_mature_osn,
    min_counts=1000,
    max_pct_mt=10
)

print(f"Cells after QC filtering: {adata_mature_osn.n_obs}")

Cells after QC filtering: 18735


Re-normalize and identify HVGs again

In [162]:
adata_mature_osn = preprocessing.normalize_and_select_hvgs(
    adata_mature_osn,
    n_top_genes=3000
)

Redo PCA with only 20 PCs

In [163]:
# Dimensional reduction of mature OSNs
adata_mature_osn = dr.run_pca(
    adata_mature_osn,
    n_comps=20
)

adata_mature_osn = dr.run_umap(
    adata_mature_osn,
    n_pcs=20
)

adata_mature_osn = dr.leiden_clustering(
    adata_mature_osn,
    resolution=1.0
)

/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/dim_reduction.py:12: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  n_comps : int, optional
/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/dim_reduction.py:49: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
  Annotated single-cell dataset.


Annotate immature neurons using marker genes

In [164]:
adata_mature_osn = annotation.score_gene_sets(
    adata_mature_osn,
    {
        "immature_OSN_score": markers.immature_osn_markers
    }
)

plotting.plot_marker_umap(
    adata_mature_osn,
    genes=[
        "immature_OSN_score",
        "Gap43",
        "Sox11",
        "Calb2",
        "pct_counts_mt",
        "total_counts"
    ],
    cmap="viridis_r",
    filename="mature_osn_immature_score_umaps.png"
)

Identify immature clusters

In [165]:
clusters_to_remove, mature_cluster_scores = (
    annotation.identify_clusters_by_score(
        adata_mature_osn,
        score_key="immature_OSN_score",
        threshold=0.2
    )
)

print("\nClusters with immature OSN score > 0.2 (remove):")
print(clusters_to_remove)


Clusters with immature OSN score > 0.2 (remove):
['7']


/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/annotation.py:192: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


Remove remaining immature OSN clusters

In [166]:
# Remove immature-like OSN clusters
adata_mature_osn = annotation.remove_clusters(
    adata_mature_osn,
    clusters_to_remove
)

print(
    f"Remaining mature OSN cells: {adata_mature_osn.n_obs}"
)


# Calculate percentage of original cells retained as mature OSNs
n_original_cells = adata_all.n_obs
n_mature_osn_cells = adata_mature_osn.n_obs

percent_mature_osns = (
    n_mature_osn_cells / n_original_cells
) * 100

print(
    f"Mature OSNs represent {percent_mature_osns:.1f}% "
    f"of the original cell population "
    f"({n_mature_osn_cells}/{n_original_cells} cells)"
)

Remaining mature OSN cells: 17676
Mature OSNs represent 51.9% of the original cell population (17676/34067 cells)


## Analyze differential expression in just mature OSNs

Plot mature OSN UMAP colored by experimental groups

In [167]:
# Plot mature OSN UMAPs
plotting.plot_umap_clusters(
    adata_mature_osn,
    filename="mature_osn_umap_leiden_clusters.png"
)

plotting.plot_umap_metadata(
    adata_mature_osn,
    colors=["sample", "condition"],
    filename="mature_osn_umap_metadata.png"
)

Get the most differentially expressed genes across conditions

In [168]:
# Find marker genes by condition
adata_mature_osn = expression.rank_cluster_markers(
    adata_mature_osn,
    groupby="condition"
)

# Get top 20 markers
top_genes = expression.get_top_marker_genes(
    adata_mature_osn,
    n_genes=20
)

print("Top DE gene for each condition:")
for group, genes in top_genes.items():
    print(f"  {group}: {genes[0]}")

# Save results
expression.save_top_genes_csv(
    top_genes,
    filename="mature_osn_top20_DE_genes_by_condition.csv"
)

Top DE gene for each condition:
  Diestrus: Fezf1
  Proestrus: Gng13


'../results/mature_osn_top20_DE_genes_by_condition.csv'

In [169]:
# Use volcano-style filtering
# Get DE results
de_results = sc.get.rank_genes_groups_df(
    adata_mature_osn,
    group="Proestrus"
)

# View strongest DE genes
de_results[
    (de_results["pvals_adj"] < 0.05) &
    (abs(de_results["logfoldchanges"]) > 0.25)
].head(50)


# Volcano plot
plotting.plot_volcano(
    de_results,
    filename="mature_osn_proestrus_vs_diestrus_volcano.png",
    n_labels=20
)

## Analyze differential expression in Bowman's gland cells

Score Bowman's gland markers

In [170]:
# Score Bowman's gland markers
adata_all = annotation.score_gene_sets(
    adata_all,
    {
        "bowmans_gland_score": markers.bowmans_gland_markers
    }
)

# Plot Bowman's gland score
plotting.plot_umap_clusters(
    adata_all,
    cluster_key="bowmans_gland_score",
    filename="bowmans_gland_score_umap.png"
)

Identify Bowman's gland clusters

In [171]:
bowmans_gland_clusters, bowmans_cluster_scores = (
    annotation.identify_clusters_by_score(
        adata_all,
        score_key="bowmans_gland_score",
        threshold=0.6
    )
)

print("Bowman's gland clusters (score > 0.6):")
print(bowmans_gland_clusters)

# Plot Leiden clusters and Bowman's gland score
plotting.plot_umap_metadata(
    adata_all,
    colors=["leiden", "bowmans_gland_score"],
    filename="bowmans_gland_clusters.png"
)

# Subset Bowman's gland cells
adata_bowmans = annotation.subset_clusters(
    adata_all,
    bowmans_gland_clusters
)

print(f"Bowman's gland cells: {adata_bowmans.n_obs}")

/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/annotation.py:192: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


Bowman's gland clusters (score > 0.6):
['12', '13', '15']
Bowman's gland cells: 1021


In [172]:
print("Bowman's gland cells by condition:")
display(
    adata_bowmans.obs["condition"].value_counts()
)

print("\nBowman's gland cells by sample:")
display(
    adata_bowmans.obs["sample"].value_counts()
)

Bowman's gland cells by condition:


condition
Proestrus    529
Diestrus     492
Name: count, dtype: int64


Bowman's gland cells by sample:


sample
D1    360
P1    339
P2    190
D2    132
Name: count, dtype: int64

See if Bowman's gland cells are differentially expressed between conditions

In [173]:
# Find DE genes by condition
adata_bowmans = expression.rank_cluster_markers(
    adata_bowmans,
    groupby="condition"
)

# Extract Proestrus DE results
bowmans_condition_de = sc.get.rank_genes_groups_df(
    adata_bowmans,
    group="Proestrus"
)

display(bowmans_condition_de.head(20))

,names,scores,logfoldchanges,pvals,pvals_adj
0,Rps29,7.763225,0.438348,8.279659e-15,2.070163e-10
1,Gfy,7.042339,1.523784,1.890394e-12,1.991493e-08
2,Gng13,7.009629,1.147405,2.389505e-12,1.991493e-08
3,Rpl38,6.952600,0.412958,3.586149e-12,2.241612e-08
4,Calm1,6.256033,0.782889,3.948918e-10,1.645580e-06
5,Stoml3,6.172560,1.399623,6.719288e-10,2.400034e-06
6,Hsp90ab1,6.003808,0.250611,1.927421e-09,6.023914e-06
7,Chga,5.959204,1.446170,2.534689e-09,7.041649e-06
8,Atp5k,5.853110,0.607722,4.824633e-09,1.104372e-05
9,Ubb,5.851943,0.263184,4.858652e-09,1.104372e-05


Perform dimensionality reduction and plot

In [174]:
# Dimensionality reduction for Bowman's gland cells
adata_bowmans = dr.run_dimensionality_reduction(
    adata_bowmans,
    n_top_genes=2000,
    n_pcs=20
)

# Plot UMAP by experimental groups
plotting.plot_umap_metadata(
    adata_bowmans,
    colors=["condition", "sample"],
    filename="bowmans_gland_umap_metadata.png"
)

/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/dim_reduction.py:72: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  def leiden_clustering(
